# SHAP (SHapley Additive exPlanations) - Interactive Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alan-turing-institute/tea-techniques/blob/main/tutorials/notebooks/shap/shap-tutorial.ipynb)

**Assurance Goal:** Explainability  
**Difficulty:** Intermediate  
**Time:** ~45 minutes

---

## 1. Overview

### What is SHAP?

SHAP (SHapley Additive exPlanations) is a technique that explains individual predictions by computing how much each feature contributed to that prediction. It's based on **Shapley values** from cooperative game theory—a mathematically principled way to fairly distribute credit among team members.

You can think of it as follows: if a team wins a prize, how do you fairly divide it based on each member's contribution? Shapley values solve this problem by considering all possible team combinations and measuring each member's marginal contribution.

### The "Before and After" Transformation

Without SHAP:
```
Input: [8 features about a house] → Model → Output: £354,000
```
*Why this prediction? We have no idea.*

With SHAP:
```
Input: [8 features] → Model → Output: £354,000
                                      ↓
                             SHAP Decomposition:
                             • Median Income: +£82,000
                             • House Age: -£15,000  
                             • Avg Rooms: +£23,000
                             • Location: +£45,000
                             • ... (other features)
                             • Base value: £219,000
                             ─────────────────────
                             = £354,000 ✓
```
*Now we can begin to explain exactly why.*

### When should I use SHAP?

- **Model debugging**: "Why is my model making this unexpected prediction?"
- **Regulatory compliance**: "We need to explain decisions to regulators."
- **Stakeholder communication**: "Why was this loan denied?"
- **Feature importance**: "Which features matter most, globally and locally?"
- **Explainability assurance cases**: "We need evidence that our model is explainable."

### Key Concepts You'll Learn in This Tutorial

- **SHAP values**: How much each feature contributes to a prediction (can be positive or negative)
- **Base value**: The average prediction across your training data - SHAP values are *relative* to this
- **Local vs Global**: SHAP provides both individual and aggregate explanations
- **Explainer types**: Different algorithms optimised for different model types

## 2. Prerequisites

### Required Knowledge

- Basic Python programming
- Familiarity with scikit-learn style ML workflow (fit, predict)
- Understanding of what features and predictions are

### Setup

Run the cell below to install required packages:

In [ ]:
# Install required packages (uncomment if running in Colab)
# !pip install -q shap>=0.42.0 xgboost>=1.5.0 scikit-learn>=1.0.0 matplotlib>=3.4.0 pandas>=1.3.0

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Initialise SHAP JS visualisations for notebooks
# Note: If plots don't render, try using matplotlib=True in plot commands
# or ensure you're using a compatible Jupyter environment
try:
    shap.initjs()
except Exception:
    print("Note: shap.initjs() failed - some interactive plots may not render.")
    print("Static matplotlib plots will still work.")

print(f"SHAP version: {shap.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print("Setup complete!")

### Dataset: California Housing

We'll use the California Housing dataset - a classic ML dataset where we predict median house values based on features like:
- **MedInc**: Median income in the block group
- **HouseAge**: Median age of houses in the block
- **AveRooms**: Average number of rooms per household
- **AveBedrms**: Average number of bedrooms per household
- **Population**: Block group population
- **AveOccup**: Average number of household members
- **Latitude/Longitude**: Location coordinates

**Why this dataset?** House prices are intuitive - we all have intuitions about what makes a house expensive. This makes it easier to validate whether SHAP explanations make sense.

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target  # Median house value in $100,000s

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"\nFeatures: {list(X.columns)}")
print(f"\nTarget: Median house value (in $100,000s)")
print(f"  Mean: ${y.mean() * 100000:,.0f}")
print(f"  Range: ${y.min() * 100000:,.0f} - ${y.max() * 100000:,.0f}")

In [ ]:
# Quick look at the data
X_train.head()

## 3. How It Works

### Intuition: The Coalition Game

Imagine you're a project manager and your team just won a £1,000 bonus. The team members are:
- Alice (data analyst)
- Bob (developer)  
- Carol (designer)

How do you fairly divide the bonus based on each person's contribution?

**The Shapley approach:**
1. Consider every possible team combination
2. For each combination, measure the marginal contribution when each person joins
3. Average each person's marginal contributions across all orderings

This gives a mathematically "fair" allocation that satisfies key properties like:
- If you contribute nothing, you get nothing
- If two people contribute equally, they get equal shares
- The shares sum to the total prize

### Interactive Example: Manual Shapley Calculation

Let's calculate Shapley values by hand for a simple 3-feature prediction.

Suppose we have a tiny model that predicts house price using just 3 features:
- **A** = Income level
- **B** = House size
- **C** = Location score

And we can measure how the prediction changes with different feature combinations:

In [ ]:
# Simulated predictions for different feature subsets
# (In practice, "missing" features use average/baseline values)

predictions = {
    frozenset():        200,   # No features (baseline)
    frozenset(['A']):   280,   # Only Income
    frozenset(['B']):   230,   # Only Size
    frozenset(['C']):   220,   # Only Location
    frozenset(['A','B']): 320, # Income + Size
    frozenset(['A','C']): 310, # Income + Location
    frozenset(['B','C']): 260, # Size + Location
    frozenset(['A','B','C']): 350,  # All features (full prediction)
}

print("Predictions with different feature combinations (in £1000s):")
for features, pred in predictions.items():
    feature_str = ', '.join(sorted(features)) if features else '(none)'
    print(f"  {feature_str:15} → £{pred}k")

In [ ]:
# Calculate Shapley value for feature A (Income)
# We consider all orderings and measure A's marginal contribution

from math import factorial
from itertools import permutations

def marginal_contribution(feature, coalition, predictions):
    """Calculate marginal contribution of adding 'feature' to 'coalition'"""
    with_feature = predictions[frozenset(coalition | {feature})]
    without_feature = predictions[frozenset(coalition)]
    return with_feature - without_feature

features = ['A', 'B', 'C']

print("Calculating Shapley value for feature A (Income):\n")
print("Coalition → A joins → Marginal Contribution")
print("-" * 50)

shapley_contributions = []
for other_features in [set(), {'B'}, {'C'}, {'B', 'C'}]:
    mc = marginal_contribution('A', other_features, predictions)
    coalition_str = ', '.join(sorted(other_features)) if other_features else '(empty)'
    shapley_contributions.append(mc)
    print(f"{coalition_str:15} → A joins → £{mc:+}k")

shapley_A = np.mean(shapley_contributions)
print(f"\nShapley value for A = average = £{shapley_A:.1f}k")

In [ ]:
# Calculate all Shapley values
def calculate_shapley(feature, all_features, predictions):
    """Calculate exact Shapley value for a feature"""
    other_features = [f for f in all_features if f != feature]
    n = len(all_features)
    shapley_value = 0

    # Iterate over all possible coalitions (subsets of other features)
    from itertools import combinations
    for size in range(len(other_features) + 1):
        for coalition in combinations(other_features, size):
            coalition_set = set(coalition)
            mc = marginal_contribution(feature, coalition_set, predictions)
            # Weight by coalition size
            weight = factorial(len(coalition_set)) * factorial(n - len(coalition_set) - 1) / factorial(n)
            shapley_value += weight * mc

    return shapley_value

print("Shapley Values (manual calculation):")
print("=" * 40)
shapley_values = {}
for f in features:
    sv = calculate_shapley(f, features, predictions)
    shapley_values[f] = sv
    print(f"  {f} (Income/Size/Location): £{sv:.1f}k")

print(f"\nBase value (no features): £{predictions[frozenset()]}k")
print(f"Sum of Shapley values: £{sum(shapley_values.values()):.1f}k")
print(f"Full prediction: £{predictions[frozenset(features)]}k")
print(f"\nCheck: Base + Shapley values = {predictions[frozenset()]} + {sum(shapley_values.values()):.1f} = {predictions[frozenset()] + sum(shapley_values.values()):.1f}k ✓")

### The Key Insight: Additivity

Notice that **base value + all SHAP values = full prediction**. This is guaranteed by the mathematics!

$$\text{Prediction} = \text{Base Value} + \sum_{i=1}^{n} \text{SHAP}_i$$

This "additivity" property means we can decompose *any* prediction into interpretable parts.

<details>
<summary><b>Mathematical Foundations (click to expand)</b></summary>

### The Shapley Value Formula

For a feature $i$ with $n$ total features:

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|! (n - |S| - 1)!}{n!} [f(S \cup \{i\}) - f(S)]$$

Where:
- $N$ is the set of all features
- $S$ is a subset of features not including $i$
- $f(S)$ is the model prediction using only features in $S$
- The fraction is a weighting factor based on coalition size

### Computational Challenge

For $n$ features, we need to evaluate $2^n$ feature subsets. With 20 features, that's over 1 million evaluations per prediction!

SHAP solves this with efficient approximations:
- **TreeSHAP**: Exact, polynomial-time algorithm for tree models
- **KernelSHAP**: Approximation via weighted linear regression for any model

</details>

### The Base Value: A Critical Concept

**SHAP values are always relative to the base value (expected prediction).**

This means:
- A SHAP value of +£50,000 means "this feature adds £50,000 *compared to the average house*"
- A SHAP value of -£10,000 means "this feature reduces the predicted value by £10,000 *from the average*"

If the average house costs £200,000, a house predicted at £150,000 might have:
- Small positive SHAP values (features slightly above average)
- Large negative SHAP values (features that push the price down)

**Common mistake**: Thinking a small negative SHAP value means the feature makes the house "cheap". It just means that feature contributes less than average - the house could still be expensive!

### How SHAP Handles Correlated Features (IMPORTANT!)

A critical nuance: **different SHAP algorithms handle correlated features differently**.

**TreeSHAP (path-dependent, default for tree models):**
- Respects the correlations captured by the tree structure
- When features are correlated, credit gets "split" between them
- Example: If Income and Education are highly correlated, each might get ~50% of the credit that would go to just Income alone

**KernelSHAP / Interventional TreeSHAP:**
- Breaks feature dependencies by assuming independence when marginalising
- Can produce different attributions than path-dependent TreeSHAP

**What this means for assurance:**
- With correlated features, identifying the "true driver" is difficult
- Credit splitting is mathematically correct but may not match human intuition
- Always check feature correlations and document them in your evidence!

## 4. Hands-On Implementation

### 4.1 Train a Model

First, let's train an XGBoost regressor. We'll use this model for all our SHAP explanations.

In [ ]:
# Train XGBoost model
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model Performance:")
print(f"  RMSE: ${rmse * 100000:,.0f}")
print(f"  R²: {r2:.3f}")
print(f"\nThe model explains {r2*100:.1f}% of the variance in house prices.")

### 4.2 Basic Usage - TreeExplainer

For tree-based models (XGBoost, LightGBM, CatBoost, Random Forest), use `TreeExplainer` - it's exact and fast.

In [ ]:
# Create TreeExplainer
# We pass background data to establish the base value
# Using 500 samples for stability (Yuan et al. recommend 500-1000)
background = shap.sample(X_train, 500, random_state=42)

# IMPORTANT: Passing data= forces "Interventional" TreeSHAP
# - This ASSUMES feature independence when marginalising
# - Without data=, XGBoost uses "Path-Dependent" TreeSHAP (respects correlations)
# - We use Interventional here for stability, but note this distinction for assurance!
explainer = shap.TreeExplainer(model, data=background)

print(f"Explainer created with {len(background)} background samples")
print(f"Base value (expected prediction): ${explainer.expected_value * 100000:,.0f}")
print(f"\nNote: Using Interventional TreeSHAP (assumes feature independence)")
print("For path-dependent explanations, omit the data= parameter.")

In [ ]:
# Compute SHAP values for test set
# This may take a minute...
shap_values = explainer(X_test)

print(f"SHAP values computed for {len(X_test)} test samples")
print(f"Shape: {shap_values.values.shape}")
print(f"  - {shap_values.values.shape[0]} samples")
print(f"  - {shap_values.values.shape[1]} features")

In [ ]:
# Verify additivity: base_value + sum(shap_values) = prediction
idx = 0  # First test sample
base = shap_values.base_values[idx]
shap_sum = shap_values.values[idx].sum()
prediction = model.predict(X_test.iloc[[idx]])[0]
computed_total = base + shap_sum

print("Additivity Check (first test sample):")
print(f"  Base value:        ${base * 100000:>12,.0f}")
print(f"  Sum of SHAP:       ${shap_sum * 100000:>12,.0f}")
print(f"  ─────────────────────────────────")
print(f"  Total:             ${computed_total * 100000:>12,.0f}")
print(f"  Model prediction:  ${prediction * 100000:>12,.0f}")

# Use np.isclose for precise verification (handles floating-point errors)
residual = abs(computed_total - prediction)
is_match = np.isclose(computed_total, prediction, rtol=1e-5)
print(f"\n  Residual: {residual:.2e} (effectively zero)")
print(f"  ✓ Match verified with np.isclose(rtol=1e-5): {is_match}")

### 4.3 Interpreting Results

#### Local Explanations: Explaining Individual Predictions

**Waterfall Plot**: Shows how each feature pushes the prediction from the base value.

In [ ]:
# Waterfall plot for a single prediction
sample_idx = 0

print(f"Explaining prediction for test sample {sample_idx}:")
print(f"  Actual value: ${y_test.iloc[sample_idx] * 100000:,.0f}")
print(f"  Predicted value: ${model.predict(X_test.iloc[[sample_idx]])[0] * 100000:,.0f}")
print("\nFeature values for this house:")
for col in X_test.columns:
    print(f"  {col}: {X_test.iloc[sample_idx][col]:.2f}")

In [ ]:
# Waterfall plot
# Reading the plot:
# - E[f(X)] at bottom: base value (average prediction)
# - f(x) at top: final prediction
# - Red bars: features that INCREASE the prediction
# - Blue bars: features that DECREASE the prediction
# - Bar length: magnitude of contribution

shap.plots.waterfall(shap_values[sample_idx], max_display=10, show=False)
plt.title("Waterfall Plot: How Features Contribute to This Prediction\n"
          "(Red = increases price, Blue = decreases price)", fontsize=10)
plt.tight_layout()
plt.show()

**Reading the Waterfall Plot:**

- Start at the **bottom**: This is E[f(X)], the base value (~average prediction)
- Move **upward**: Each bar shows a feature's contribution
- **Red bars** push the prediction **higher** (positive contribution)
- **Blue bars** push the prediction **lower** (negative contribution)
- End at the **top**: f(x), the final prediction
- The **number** next to each feature is its actual value for this sample

#### Exercise: Explain an Expensive House

Let's find and explain one of the most expensive houses in our test set.

In [ ]:
# Find an expensive house
predictions = model.predict(X_test)
expensive_idx = np.argmax(predictions)

print(f"Most expensive predicted house (index {expensive_idx}):")
print(f"  Predicted: ${predictions[expensive_idx] * 100000:,.0f}")
print(f"  Actual: ${y_test.iloc[expensive_idx] * 100000:,.0f}")

shap.plots.waterfall(shap_values[expensive_idx], max_display=10)

**Question**: Looking at the waterfall plot above, which features contribute most to making this house expensive? Does this match your intuition about what makes houses valuable?

#### Global Explanations: Understanding Overall Model Behaviour

**Beeswarm Plot**: Shows the distribution of SHAP values for all samples.

In [ ]:
# Beeswarm plot
# Reading the plot:
# - Each dot is one sample
# - X-axis: SHAP value (how much this feature affects prediction)
# - Colour: feature value (red = high, blue = low)
# - Features sorted by importance (most important at top)

shap.plots.beeswarm(shap_values, max_display=10, show=False)
plt.title("Beeswarm Plot: Feature Importance Distribution\n"
          "(Each dot = one sample, Colour = feature value)", fontsize=10)
plt.tight_layout()
plt.show()

**Reading the Beeswarm Plot:**

- Each **dot** represents one sample
- **X-axis**: SHAP value (contribution to prediction)
- **Colour**: Feature value (red = high, blue = low)
- **Vertical stacking**: Density of samples at that SHAP value
- Features are **sorted by importance** (most impactful at top)

**What to look for:**
- MedInc (income): Red dots on right = high income → higher prices ✓
- Latitude: The relationship with location (latitude affects coastal proximity)
- HouseAge: Does old or new matter more for price?

In [ ]:
# Bar plot: Mean absolute SHAP values (global feature importance)
shap.plots.bar(shap_values, max_display=10, show=False)
plt.title("Global Feature Importance\n(Mean |SHAP value| across all samples)", fontsize=10)
plt.tight_layout()
plt.show()

#### Feature Interactions: Dependence Plots

**Dependence plots** show how one feature's SHAP value varies with its value, coloured by an interacting feature.

In [ ]:
# Dependence plot for MedInc (most important feature)
# X-axis: Feature value
# Y-axis: SHAP value for that feature
# Colour: Automatically chosen interacting feature

shap.plots.scatter(shap_values[:, "MedInc"], color=shap_values[:, "AveOccup"], show=False)
plt.title("How Median Income Affects Predictions\n"
          "(Colour = Average Occupancy)", fontsize=10)
plt.tight_layout()
plt.show()

### 4.4 Common Pitfalls

#### Pitfall 1: Small Background Dataset

Research by Yuan et al. (2022) shows SHAP explanations can be unstable with small background datasets.

In [ ]:
# Demonstrate instability with small background
print("Comparing SHAP values with different background sizes...\n")

sample_idx = 0
results = []

for bg_size in [50, 100, 500]:
    bg = shap.sample(X_train, bg_size, random_state=42)
    exp = shap.TreeExplainer(model, data=bg)
    sv = exp(X_test.iloc[[sample_idx]])

    # Get SHAP value for MedInc (most important feature)
    medinc_shap = sv.values[0][0]  # First feature is MedInc
    results.append((bg_size, medinc_shap))
    print(f"Background size {bg_size:3d}: MedInc SHAP = {medinc_shap:.4f}")

print(f"\nVariation: {max(r[1] for r in results) - min(r[1] for r in results):.4f}")
print("\nRecommendation: Use 500+ background samples for stability.")

#### Pitfall 2: Correlated Features

Let's check the correlations in our dataset:

In [ ]:
# Check feature correlations
corr_matrix = X_train.corr()

# Find highly correlated pairs (|r| > 0.5)
print("Highly correlated feature pairs (|r| > 0.5):")
print("=" * 50)
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr_matrix.columns[i]:12} ↔ {corr_matrix.columns[j]:12}: r = {r:.2f}")

print("\nWhen features are correlated:")
print("  - SHAP credit gets 'split' between them")
print("  - Hard to identify the 'true driver'")
print("  - Document correlations in your assurance evidence!")

#### Pitfall 3: Confusing Correlation with Causation

SHAP values show **associations**, not causal effects!

Example: Latitude has high SHAP importance. This doesn't mean "moving a house north will change its price." It means the model has learnt that latitude is associated with price (likely because it correlates with proximity to expensive coastal areas).

### 4.5 Testing Explanation Stability

This is **critical for assurance** - we need to know if our explanations are reliable.

In [ ]:
def test_shap_stability(model, X_train, X_test, n_runs=3, background_size=500):
    """
    Test SHAP explanation stability across multiple runs with different
    background samples. This is a 'unit test for explainability'.
    """
    feature_names = X_test.columns.tolist()
    n_features = len(feature_names)
    n_samples = len(X_test)

    # Store SHAP values from each run
    all_shap_values = []

    print(f"Running {n_runs} SHAP computations with different backgrounds...")
    for run in range(n_runs):
        # Different random background each time
        bg = shap.sample(X_train, background_size, random_state=run*100)
        exp = shap.TreeExplainer(model, data=bg)
        sv = exp(X_test)
        all_shap_values.append(sv.values)
        print(f"  Run {run+1}/{n_runs} complete")

    # Stack and compute statistics
    all_shap_values = np.stack(all_shap_values, axis=0)  # (n_runs, n_samples, n_features)

    # Compute mean and std across runs for each sample/feature
    mean_shap = np.mean(all_shap_values, axis=0)
    std_shap = np.std(all_shap_values, axis=0)

    # Compute stability metrics per feature (averaged across samples)
    feature_mean_abs_shap = np.mean(np.abs(mean_shap), axis=0)
    feature_mean_std = np.mean(std_shap, axis=0)

    # Coefficient of variation (relative stability)
    cv = feature_mean_std / (feature_mean_abs_shap + 1e-10)

    # Compile results
    stability_results = []
    for i, feat in enumerate(feature_names):
        stability_results.append({
            'feature': feat,
            'mean_abs_shap': feature_mean_abs_shap[i],
            'mean_std': feature_mean_std[i],
            'cv': cv[i],
            'stability': 'high' if cv[i] < 0.1 else ('medium' if cv[i] < 0.3 else 'low')
        })

    # Sort by importance
    stability_results = sorted(stability_results, key=lambda x: -x['mean_abs_shap'])

    return stability_results, all_shap_values

# Run stability test
stability_results, all_shap_arrays = test_shap_stability(
    model, X_train, X_test.head(100), n_runs=3, background_size=500
)

In [ ]:
# Display stability results
print("\nSHAP Explanation Stability Report")
print("=" * 70)
print(f"{'Feature':<15} {'Mean |SHAP|':>12} {'Std':>10} {'CV':>8} {'Stability':>10}")
print("-" * 70)

for r in stability_results:
    stability_emoji = {'high': '🟢', 'medium': '🟡', 'low': '🔴'}[r['stability']]
    print(f"{r['feature']:<15} {r['mean_abs_shap']:>12.4f} {r['mean_std']:>10.4f} "
          f"{r['cv']:>8.2%} {stability_emoji} {r['stability']:>6}")

print("\nInterpretation:")
print("  CV (Coefficient of Variation) = Std / Mean")
print("  🟢 High stability: CV < 10% - SHAP values are consistent")
print("  🟡 Medium stability: CV 10-30% - Some variation, interpret with care")
print("  🔴 Low stability: CV > 30% - High variation, may be unreliable")

## 5. Generating Assurance Evidence

### 5.1 The "Failed Audit" Scenario

> *"An auditor rejects your model because Feature A and Feature B swap importance rankings randomly between runs. The explanation is unstable and therefore unsuitable as assurance evidence. Here's how to detect and address this..."*

This is why stability testing (Section 4.5) matters! Let's see what proper assurance evidence looks like.

### 5.2 What Claims Can SHAP Support?

| Claim | SHAP Support | Notes |
|-------|--------------|-------|
| ✅ "The model uses feature X in making predictions" | Strong | SHAP directly measures feature contribution |
| ✅ "Feature X has positive/negative association with outcome" | Strong | Direction of SHAP values shows this |
| ✅ "This specific prediction was driven by features X, Y, Z" | Strong | Local explanations show this exactly |
| ⚠️ "The model is fair" | Insufficient | SHAP can show *what* features are used, not whether their use is *appropriate* |
| ❌ "Feature X causes the outcome" | Not supported | SHAP shows correlation/association, not causation |

### 5.3 Generating Evidence Artefacts

Let's generate concrete, exportable evidence for an assurance case.

In [ ]:
def generate_shap_evidence_artefact(
    model,
    X_train,
    X_test,
    model_id="xgb_housing_v1",
    background_size=500,
    stability_runs=3,
    stability_threshold=0.1  # CV threshold for PASS/FAIL
):
    """
    Generate a complete SHAP evidence artefact suitable for assurance cases.

    Returns a dictionary that can be saved as JSON.
    Includes PASS/FAIL status for CI/CD integration.
    """
    # Run stability analysis
    stability_results, _ = test_shap_stability(
        model, X_train, X_test.head(100),
        n_runs=stability_runs,
        background_size=background_size
    )

    # Check correlations
    corr_matrix = X_train.corr()
    high_correlations = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            r = corr_matrix.iloc[i, j]
            if abs(r) > 0.5:
                high_correlations.append({
                    'feature_1': corr_matrix.columns[i],
                    'feature_2': corr_matrix.columns[j],
                    'correlation': round(r, 3)
                })

    # Determine overall stability and PASS/FAIL status
    top_3_stable = all(r['cv'] < stability_threshold for r in stability_results[:3])
    overall_stability = 'high' if all(r['stability'] == 'high' for r in stability_results[:3]) else \
                        ('medium' if any(r['stability'] == 'high' for r in stability_results[:3]) else 'low')

    # PASS/FAIL gate for assurance
    stability_pass = top_3_stable and overall_stability in ['high', 'medium']

    # Build evidence artefact
    evidence = {
        'metadata': {
            'model_id': model_id,
            'timestamp': datetime.now().isoformat(),
            'shap_version': shap.__version__,
            'explainer_type': 'TreeExplainer (Interventional)',
            'background_size': background_size,
            'test_samples_analysed': len(X_test.head(100))
        },
        'assurance_gate': {
            'status': 'PASS' if stability_pass else 'FAIL',
            'stability_threshold_cv': stability_threshold,
            'top_3_features_below_threshold': top_3_stable,
            'message': 'Explanations are stable and suitable for assurance evidence.' if stability_pass
                      else 'WARNING: Explanation stability below threshold. Review before using as evidence.'
        },
        'feature_ranking': [
            {
                'rank': i + 1,
                'feature': r['feature'],
                'mean_abs_shap': round(r['mean_abs_shap'], 4),
                'stability': r['stability'],
                'coefficient_of_variation': round(r['cv'], 4)
            }
            for i, r in enumerate(stability_results)
        ],
        'stability_analysis': {
            'n_runs': stability_runs,
            'background_size': background_size,
            'overall_stability': overall_stability,
            'top_3_features_stable': all(r['stability'] in ['high', 'medium'] for r in stability_results[:3])
        },
        'base_value': {
            'value': round(float(explainer.expected_value), 4),
            'interpretation': 'Average model prediction - SHAP values are relative to this baseline'
        },
        'limitations': [
            'SHAP values show associations, not causal effects',
            'Interventional TreeSHAP assumes feature independence when marginalising',
            f'Background dataset of {background_size} samples used for baseline',
            'Categorical features (if any) may have diluted importance due to one-hot encoding'
        ],
        'correlated_features': high_correlations if high_correlations else 'None with |r| > 0.5',
        'recommendations': []
    }

    # Add recommendations based on findings
    if high_correlations:
        evidence['recommendations'].append(
            f"Note: {len(high_correlations)} highly correlated feature pairs detected. "
            "SHAP credit may be split between these features."
        )

    if not stability_pass:
        evidence['recommendations'].append(
            "CRITICAL: Stability check failed. Consider increasing background dataset size "
            "or investigating why explanations are unstable before using as assurance evidence."
        )

    return evidence

# Generate the evidence artefact
evidence = generate_shap_evidence_artefact(
    model, X_train, X_test,
    model_id="xgb_california_housing_v1"
)

In [ ]:
# Display the evidence artefact
print("SHAP Evidence Artefact")
print("=" * 60)
print(json.dumps(evidence, indent=2))

In [ ]:
# Save to file (in practice)
# with open('shap_evidence.json', 'w') as f:
#     json.dump(evidence, f, indent=2)

print("\n📋 This JSON artefact can be:")
print("   - Saved as evidence for an assurance case")
print("   - Versioned alongside model artefacts")
print("   - Referenced in audit documentation")
print("   - Compared across model versions")

### 5.4 Assurance Case Integration

**Example Claim**: "The model's predictions are explainable at both individual and aggregate levels."

**Evidence Structure**:

```
Claim: Model predictions are explainable
├── Sub-claim: Individual predictions can be decomposed into feature contributions
│   └── Evidence: SHAP waterfall plots showing additive decomposition
├── Sub-claim: Global feature importance is stable and interpretable  
│   └── Evidence: Stability analysis showing CV < 10% for top features
└── Sub-claim: Explanations are documented with known limitations
    └── Evidence: SHAP evidence artefact with correlation warnings
```

**Confidence Qualifiers**:
- "Under the condition that background dataset has 500+ samples..."
- "Noting that features MedInc and AveOccup show correlation (r=0.65)..."
- "With the caveat that SHAP values show associations, not causal effects..."

### 5.5 Limitations for Assurance

When using SHAP for assurance purposes, document these limitations:

1. **Computational, not cognitive explanations**: SHAP tells you *how the model computes*, not whether this matches human reasoning

2. **Doesn't validate correctness**: SHAP explains *what* the model does, not whether it's *right*

3. **Background dataset affects results**: Document your choice and test stability

4. **Interventional vs Path-Dependent**: Passing `data=` to TreeExplainer assumes feature independence; omitting it respects correlations captured by the tree

5. **Model-agnostic methods have approximation error**: KernelSHAP is approximate; TreeSHAP is exact for trees

6. **Categorical features (One-Hot Encoding)**: When categorical features are one-hot encoded, SHAP credit is split across dummy columns. Sum the SHAP values for all columns belonging to a single category to get the true feature importance.

7. **Adversarial vulnerability**: SHAP values can potentially be manipulated by adversarial perturbations (relevant for security-critical systems)

## 6. Going Further

### Other SHAP Explainers

| Explainer | Best For | Speed | Exactness |
|-----------|----------|-------|--------|
| TreeExplainer | Tree models (XGBoost, RF, etc.) | Fast | Exact |
| DeepExplainer | Neural networks (TensorFlow/PyTorch) | Medium | Approximate |
| GradientExplainer | Neural networks | Medium | Approximate |
| KernelExplainer | Any model | Slow | Approximate |
| LinearExplainer | Linear models | Fast | Exact |

### Advanced Topics

- **SHAP interaction values**: Measure how pairs of features interact
- **Text explanations**: Explain NLP model predictions word-by-word
- **Image explanations**: Highlight important regions in image classifications

### Related Techniques

- **[LIME](https://alan-turing-institute.github.io/tea-techniques/techniques/local-interpretable-model-agnostic-explanations)**: Another local explanation method; fits a simple model around each prediction
- **Integrated Gradients**: Gradient-based explanations for neural networks
- **Permutation Importance**: Global importance via feature shuffling

### Additional Resources

- [SHAP Documentation](https://shap.readthedocs.io/)
- [Original SHAP Paper (Lundberg & Lee, 2017)](https://arxiv.org/abs/1705.07874)
- [Yuan et al. (2022) - SHAP Stability Study](http://arxiv.org/abs/2204.11351)
- [TEA Techniques: SHAP](https://alan-turing-institute.github.io/tea-techniques/techniques/shapley-additive-explanations)

## 7. Reflective Questions

Before finishing, consider these questions:

### Conceptual Understanding

1. **If two features are highly correlated, what happens to their SHAP values? How would you address this in an assurance context?**
   
   *Hint: Think about credit splitting and how you would document this limitation.*

2. **Why is the base value important for interpreting SHAP values?**
   
   *Hint: What does a SHAP value of +£50,000 actually mean?*

### Application Scenarios

3. **A stakeholder asks why a loan was denied. How would you use SHAP to answer, and what caveats would you communicate?**
   
   *Hint: Consider local explanations, but also what SHAP cannot tell you.*

4. **You're auditing a model and notice SHAP values change significantly with different background datasets. What does this tell you, and what would you recommend?**
   
   *Hint: Think about the stability analysis from Section 4.5.*

### Critical Thinking

5. **Can SHAP prove a model is "fair"? What additional evidence would you need?**
   
   *Hint: SHAP shows what features are used, not whether their use is appropriate.*

6. **An engineer says "SHAP proves that income causes higher prices in our model." How would you respond?**
   
   *Hint: Association vs. causation.*

---

## Summary

In this tutorial, you learnt:

- **What SHAP is**: A method to explain predictions by computing each feature's contribution based on Shapley values from game theory

- **How to apply it**: Using TreeExplainer for tree models with appropriate background datasets (500+ samples)

- **How to interpret results**: Waterfall plots for local explanations, beeswarm plots for global patterns

- **Common pitfalls**: Small background datasets, correlated features, confusing correlation with causation

- **Assurance applications**: Generating evidence artefacts, testing stability, documenting limitations

**Key Takeaways for Assurance**:
1. Always test explanation stability before using SHAP as evidence
2. Document correlated features and their impact on credit splitting
3. Be precise: SHAP shows associations, not causal effects
4. Generate structured evidence artefacts that can be versioned and audited

**Next Steps**:
- Apply SHAP to your own models
- Explore the [LIME tutorial](https://alan-turing-institute.github.io/tea-techniques/tutorials/notebooks/lime/) for comparison
- Read the original paper: Lundberg & Lee (2017)

---

*This tutorial is part of the [TEA Techniques](https://alan-turing-institute.github.io/tea-techniques/) project - helping practitioners generate assurance evidence for AI systems.*